# Regresión Lineal - Modelo Final

Predicción del precio de cierre de BTC a 7 días, usando el mismo enfoque y las mismas features del Experimento 01 (`01_experimentos_full_data(KNN,RF,LR).ipynb`): un único `LinearRegression` multi-salida (`Close_t+1` ... `Close_t+7`) entrenado sobre `Open`, `High`, `Low`, `Volume`, `Volatility`, `SMA_7` y `SMA_30`.

In [22]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [23]:
# carga y preparacion de datos
btc = pd.read_csv("../data/full_data.csv")
btc['Date'] = pd.to_datetime(btc['Date'])
btc.set_index('Date', inplace=True)

# crear targets futuros (Close_t+1 ... Close_t+7)
for i in range(1, 8):
    btc[f'Close_t+{i}'] = btc['Close'].shift(-i)
btc.dropna(inplace=True)

# features (mismas que el experimento 01)
features = ['Open', 'High', 'Low', 'Volume', 'Volatility', 'SMA_7', 'SMA_30']
targets = [f'Close_t+{i}' for i in range(1, 8)]

X = btc[features].copy()
Y = btc[targets].copy()

print("Valores nulos en X:", X.isnull().sum().sum())
print("Valores nulos en Y:", Y.isnull().sum().sum())

Valores nulos en X: 0
Valores nulos en Y: 0


In [24]:
# escalado
mapper = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), features)],
    remainder='drop'
)

# validacion temporal (7 folds)
n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

pipeline = Pipeline([('mapper', mapper), ('model', LinearRegression())])

In [25]:
# validacion cruzada temporal (train/val 80/20 dentro de cada fold, igual que en el experimento 01)
fold_results = []

for fold, (train_val_idx, test_idx) in enumerate(tscv.split(X)):
    X_train_val, X_test = X.iloc[train_val_idx], X.iloc[test_idx]
    Y_train_val, Y_test = Y.iloc[train_val_idx], Y.iloc[test_idx]

    val_size = int(len(X_train_val) * 0.2)
    X_train, X_val = X_train_val.iloc[:-val_size], X_train_val.iloc[-val_size:]
    Y_train, Y_val = Y_train_val.iloc[:-val_size], Y_train_val.iloc[-val_size:]

    pipeline.fit(X_train, Y_train)
    Y_val_pred = pipeline.predict(X_val)

    val_mae = mean_absolute_error(Y_val, Y_val_pred)
    val_mse = mean_squared_error(Y_val, Y_val_pred)
    val_rmse = np.sqrt(val_mse)
    val_mape = mean_absolute_percentage_error(Y_val, Y_val_pred) * 100

    fold_results.append({
        'fold': fold + 1,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_mape': val_mape,
    })

    print(f"Fold {fold+1}: MAE {val_mae:.2f} | RMSE {val_rmse:.2f} | MAPE {val_mape:.2f}%")

results_df = pd.DataFrame(fold_results)
print(f"\nPromedio -> MAE: {results_df['val_mae'].mean():.4f}, RMSE: {results_df['val_rmse'].mean():.4f}, MAPE: {results_df['val_mape'].mean():.2f}%")

Fold 1: MAE 139.11 | RMSE 180.51 | MAPE 3.67%
Fold 2: MAE 549.53 | RMSE 851.06 | MAPE 7.74%
Fold 3: MAE 2679.84 | RMSE 3864.16 | MAPE 7.20%
Fold 4: MAE 2344.69 | RMSE 3100.97 | MAPE 5.21%
Fold 5: MAE 968.31 | RMSE 1504.17 | MAPE 4.33%
Fold 6: MAE 1520.71 | RMSE 2333.04 | MAPE 3.36%
Fold 7: MAE 2892.50 | RMSE 3938.02 | MAPE 3.73%

Promedio -> MAE: 1584.9547, RMSE: 2253.1327, MAPE: 5.04%


In [26]:
# entrenamos el modelo final con todo el historico disponible
pipeline_final = Pipeline([('mapper', mapper), ('model', LinearRegression())])
pipeline_final.fit(X, Y)

Y_pred_full = pipeline_final.predict(X)
mae_full = mean_absolute_error(Y, Y_pred_full)
rmse_full = np.sqrt(mean_squared_error(Y, Y_pred_full))
mape_full = mean_absolute_percentage_error(Y, Y_pred_full) * 100

print(f"Metricas sobre todo el historico -> MAE: {mae_full:.4f}, RMSE: {rmse_full:.4f}, MAPE: {mape_full:.2f}%")

Metricas sobre todo el historico -> MAE: 1622.9000, RMSE: 2664.9051, MAPE: 4.72%


In [27]:
import os
import joblib

output_dir = "./models_final"
os.makedirs(output_dir, exist_ok=True)
joblib.dump(pipeline_final, os.path.join(output_dir, "linear_regression_multioutput.pkl"))
print(f"Modelo final guardado en {output_dir}")

Modelo final guardado en ./models_final


In [28]:
# prediccion del precio de BTC para los proximos 7 dias

raw_df = pd.read_csv("../data/full_data.csv", parse_dates=["Date"], dayfirst=False)
raw_df = raw_df.sort_values("Date").reset_index(drop=True)

last_row = raw_df.dropna(subset=features).iloc[[-1]]
last_date = last_row.iloc[0]["Date"]
last_close = float(last_row.iloc[0]["Close"])

prediction = pipeline_final.predict(last_row[features])[0]

forecast_rows = [
    {
        "horizonte": f"t+{day}",
        "Fecha futura": (last_date + pd.Timedelta(days=day)).date(),
        "Predicción precio": f"${float(prediction[day - 1]):,.0f}",
    }
    for day in range(1, 8)
]
btc_pred_lr = pd.DataFrame(forecast_rows)

print("=== Prediccion BTC a 7 dias (Regresion Lineal) ===")
print(f"Ultima fecha disponible: {last_date.date()}")
print(f"Ultimo cierre observado: USD {last_close:,.2f}")
print(btc_pred_lr.to_string(index=False))

btc_pred_lr.to_csv("prediccion_regresion_lineal_7dias.csv", index=False)
print("\nResultados guardados en: prediccion_regresion_lineal_7dias.csv")

=== Prediccion BTC a 7 dias (Regresion Lineal) ===
Ultima fecha disponible: 2026-08-24
Ultimo cierre observado: USD 78,858.93
horizonte Fecha futura Predicción precio
      t+1   2026-08-25           $78,731
      t+2   2026-08-26           $78,802
      t+3   2026-08-27           $78,887
      t+4   2026-08-28           $78,926
      t+5   2026-08-29           $78,988
      t+6   2026-08-30           $79,030
      t+7   2026-08-31           $79,028

Resultados guardados en: prediccion_regresion_lineal_7dias.csv
